# 22 · Capstone — URL Shortener + Rate Limiter

**Part 3 · Capstones** · *Prerequisites: Part 1 (esp. 01, 03, 06)* · *The classic warm-up,
done with our tools*

The URL shortener is system design's *étude*: small enough to finish, rich enough to
exercise estimation, ID generation, caching, partitioning, and — the half people skip —
protecting the whole thing with **rate limiting**. We follow nb 00 §5's framework:
requirements → workload → design → failure analysis → evolution, building every
load-bearing component.

**Step 1 — Requirements.** Functional: create a short code for a URL; redirect
`short.ly/Ab3xK9q` → the URL; optional expiry. Non-functional (numbers, not adjectives):
redirect p99 **< 50 ms**; availability **99.95%** (redirects are other people's page
loads!); codes must not be guessable-in-bulk (enumeration leaks private links).

In [ ]:
import string
import time
from collections import OrderedDict, deque

import matplotlib.pyplot as plt
import numpy as np

from sysdes import viz

viz.use_style()
rng = np.random.default_rng(seed=22)

## Step 2 — Workload (nb 01's pipeline)

In [ ]:
new_links_per_month = 100e6
read_write_ratio = 100                       # redirects dominate
avg_record_bytes = 500                       # url + code + metadata
retention_years, replication = 5, 3

writes_per_s = new_links_per_month / (30 * 86_400)
reads_per_s = writes_per_s * read_write_ratio
peak_reads = 3 * reads_per_s
stored_tb = new_links_per_month * 12 * retention_years * avg_record_bytes \
    * replication / 1e12

print(f"writes {writes_per_s:,.0f}/s · reads {reads_per_s:,.0f}/s "
      f"(peak ~{peak_reads:,.0f}/s) · storage after {retention_years}y: {stored_tb:.0f} TB")
assert writes_per_s < 100, "writes are trivial - a single partition could take them"
assert peak_reads > 10_000, "reads are the game: cache + partition the read path"
assert stored_tb < 20, "storage is modest - this is NOT a big-data problem"

The estimate *is* the architecture brief: tiny writes, hot reads, small data → one
replicated, partitioned key-value store (nb 03/05/06), a cache in front (nb 12), and all
engineering effort on the read path.

## Step 3 — Design

In [ ]:
fig, ax = viz.system_diagram(
    nodes=[
        {"id": "c", "x": 0, "y": 1, "label": "clients", "kind": "client"},
        {"id": "lb", "x": 1.9, "y": 1, "label": "load\nbalancer"},
        {"id": "api", "x": 3.8, "y": 1, "label": "api nodes\n(stateless)"},
        {"id": "rl", "x": 3.8, "y": 2.3, "label": "rate limiter", "kind": "cache"},
        {"id": "cache", "x": 6.0, "y": 1.9, "label": "LRU cache\n(code → url)", "kind": "cache"},
        {"id": "db", "x": 6.0, "y": 0.2, "label": "KV store\nhash-partitioned", "kind": "store"},
        {"id": "idgen", "x": 1.9, "y": -0.6, "label": "id generator\n(per-worker)", "kind": "service"},
    ],
    edges=[
        {"src": "c", "dst": "lb"}, {"src": "lb", "dst": "api"},
        {"src": "api", "dst": "rl", "label": "check", "bidir": True},
        {"src": "api", "dst": "cache", "label": "read-through", "bidir": True},
        {"src": "api", "dst": "db", "label": "miss / write", "bidir": True},
        {"src": "api", "dst": "idgen", "label": "on create", "style": "dashed"},
    ],
    title="Everything from Part 1, one small system",
)
plt.show()

### 3.1 ID generation — three schemes, three trade-offs

**Base62 counter.** Encode an auto-incrementing integer in `[0-9a-zA-Z]`:

In [ ]:
ALPHABET = string.digits + string.ascii_lowercase + string.ascii_uppercase   # 62 chars

def base62_encode(n: int) -> str:
    if n == 0:
        return ALPHABET[0]
    out = []
    while n:
        n, r = divmod(n, 62)
        out.append(ALPHABET[r])
    return "".join(reversed(out))

def base62_decode(s: str) -> int:
    n = 0
    for ch in s:
        n = n * 62 + ALPHABET.index(ch)
    return n

for n in (0, 61, 62, 123_456_789, 62**7 - 1):
    assert base62_decode(base62_encode(n)) == n
print(f"62^7 = {62**7:,} codes in 7 chars; id 123,456,789 -> {base62_encode(123_456_789)!r}")
assert 62**7 > 3.5e12, "7 chars outlast decades of growth"

Compact and collision-free — but sequential ids are **guessable** (crawl `/1, /2, /3...`
and harvest every private link — a real breach pattern) and, if the store is key-*range*
partitioned, all new writes hammer the last partition (nb 06 §1's timestamp disaster).
The fixes: hash-partition by code (nb 06), and don't expose raw sequence — which leads
to...

**Snowflake-style ids.** `timestamp | worker_id | sequence` packed in 64 bits:
distributed generation with no coordination (each worker owns its sub-space), roughly
time-sortable (nice for analytics), not trivially enumerable:

In [ ]:
class Snowflake:
    """41 bits of ms-timestamp | 10 bits worker | 12 bits per-ms sequence."""

    def __init__(self, worker_id: int, epoch_ms: int = 1_700_000_000_000):
        assert 0 <= worker_id < 1024
        self.worker_id = worker_id
        self.epoch = epoch_ms
        self.last_ms = -1
        self.seq = 0

    def next_id(self, now_ms: int) -> int:
        if now_ms < self.last_ms:
            # nb 08: wall clocks step backwards. Refusing is the SAFE behavior -
            # generating would risk duplicate ids.
            raise RuntimeError(f"clock moved backwards by {self.last_ms - now_ms} ms")
        if now_ms == self.last_ms:
            self.seq += 1
            if self.seq >= 4096:
                raise RuntimeError("sequence exhausted this millisecond (wait 1ms)")
        else:
            self.last_ms, self.seq = now_ms, 0
        return ((now_ms - self.epoch) << 22) | (self.worker_id << 12) | self.seq

w1, w2 = Snowflake(worker_id=1), Snowflake(worker_id=2)
t0 = 1_700_000_100_000
ids = [w1.next_id(t0), w1.next_id(t0), w1.next_id(t0 + 1),
       w2.next_id(t0), w2.next_id(t0 + 1)]
assert len(set(ids)) == len(ids), "unique across workers with ZERO coordination"
assert ids[0] < ids[1] < ids[2], "per-worker monotone; globally ~time-ordered"
try:
    w1.next_id(t0 - 500)
    raise AssertionError("must refuse")
except RuntimeError as e:
    print(f"clock skew handled loudly: {e}")
print(f"example id: {ids[0]} -> code {base62_encode(ids[0])!r}")

**Random codes.** 7 random base62 chars: unguessable, uniformly partitionable — but now
collisions are possible. Birthday arithmetic decides if that's a problem:

In [ ]:
space = 62**7
n_keys = 6e9                              # 5 years of links
p_any_collision_per_insert = n_keys / space
expected_collisions_total = n_keys**2 / (2 * space)
print(f"per-insert collision chance at {n_keys:.0e} keys: "
      f"{p_any_collision_per_insert:.2%}; expected total collisions: "
      f"{expected_collisions_total:,.0f}")
assert expected_collisions_total > 1_000, \
    "collisions WILL happen at scale - not rare enough to ignore"

Millions of collisions over the system's life — so random codes require
**check-and-retry** on insert (an atomic put-if-absent — nb 07's CAS, or a unique
constraint). Verdict: random (or snowflake-then-encode) codes + hash partitioning +
put-if-absent. Sequential counters only behind an unguessable mapping.

### 3.2 The read path — a cache and the shape of popularity

Link popularity is Zipfian: a few viral links absorb most clicks (nb 06's celebrity,
wearing a URL). That skew is the cache's best friend — a cache holding 5% of links can
serve the vast majority of traffic. Build the LRU and measure:

In [ ]:
class LRUCache:
    """OrderedDict-based LRU: hits refresh recency; inserts evict the stalest."""

    def __init__(self, capacity: int):
        self.capacity = capacity
        self.data: OrderedDict = OrderedDict()
        self.hits = self.misses = 0

    def get(self, key):
        if key in self.data:
            self.data.move_to_end(key)
            self.hits += 1
            return self.data[key]
        self.misses += 1
        return None

    def put(self, key, value):
        self.data[key] = value
        self.data.move_to_end(key)
        if len(self.data) > self.capacity:
            self.data.popitem(last=False)          # evict least-recently-used

N_LINKS = 200_000
CACHE_SHARE = 0.05
zipf_weights = 1 / np.arange(1, N_LINKS + 1) ** 1.1     # rank-frequency law
zipf_weights /= zipf_weights.sum()
requests = rng.choice(N_LINKS, size=300_000, p=zipf_weights)

cache = LRUCache(capacity=int(N_LINKS * CACHE_SHARE))
DB_MS, CACHE_MS = 8.0, 0.5
latencies = []
for code in requests:
    if cache.get(code) is None:
        cache.put(code, f"url-{code}")             # read-through on miss
        latencies.append(DB_MS + CACHE_MS)
    else:
        latencies.append(CACHE_MS)

hit_rate = cache.hits / (cache.hits + cache.misses)
lat = np.array(latencies[50_000:])                 # after warm-up
print(f"cache = {CACHE_SHARE:.0%} of keys -> hit rate {hit_rate:.0%}; "
      f"redirect p50 {np.percentile(lat, 50):.1f} ms, p99 {np.percentile(lat, 99):.1f} ms")
assert hit_rate > 0.70, "Zipf skew: a sliver of memory absorbs most of the traffic"
assert np.percentile(lat, 50) <= CACHE_MS + 0.01, "the median request never sees the DB"

The p99 still pays the DB price (the long tail of unpopular links *is* the miss
traffic) — fine, it's within budget. Note what the cache also did: it absorbed the hot
key. A viral link is millions of reads of *one* code — unpartitionable by placement
(nb 06 §3) but trivially cacheable, even in the API nodes' local memory.

### 3.3 Rate limiting — the system's immune system

Redirects are cheap for us and valuable to abusers (enumeration scans, referral fraud,
plain overload). A **rate limiter** enforces per-client budgets. Two builds:

In [ ]:
class TokenBucket:
    """capacity = burst allowance; refill_rate = sustained tokens/sec.
    Each request spends one token; empty bucket -> reject (HTTP 429)."""

    def __init__(self, capacity: float, refill_rate: float):
        self.capacity = capacity
        self.refill_rate = refill_rate
        self.tokens = capacity
        self.updated = 0.0

    def allow(self, now: float) -> bool:
        # Lazy refill: top up for the time elapsed since we last looked.
        self.tokens = min(self.capacity,
                         self.tokens + (now - self.updated) * self.refill_rate)
        self.updated = now
        if self.tokens >= 1:
            self.tokens -= 1
            return True
        return False

bucket = TokenBucket(capacity=10, refill_rate=2.0)      # burst 10, sustain 2/s
burst = sum(bucket.allow(0.0) for _ in range(15))
assert burst == 10, "burst allowance = bucket capacity, then hard stop"

# Sustained pressure on an empty bucket: 100 requests over 5s (20/s demanded).
drained = TokenBucket(capacity=10, refill_rate=2.0)
for _ in range(15):
    drained.allow(0.0)                                  # start with zero tokens
admitted = sum(drained.allow(0.05 * (i + 1)) for i in range(100))
assert admitted == 10, "sustained admission = refill_rate x time (2/s x 5s = 10)"
print(f"burst admitted: {burst}/15; sustained: {admitted}/100 over 5s (exactly 2/s)")

Token bucket is the industry default because it *permits bursts* (real clients are
bursty) while capping the average. The alternative family counts requests per window —
and the naive **fixed window** has a famous hole:

In [ ]:
LIMIT, WINDOW = 10, 1.0    # '10 requests per second'

def fixed_window_admitted(times):
    counts: dict[int, int] = {}
    admitted = []
    for t in times:
        w = int(t // WINDOW)
        counts[w] = counts.get(w, 0) + 1
        admitted.append(counts[w] <= LIMIT)
    return admitted

def sliding_log_admitted(times):
    """Exact: keep each client's timestamps; admit if < LIMIT in the last WINDOW."""
    log: deque = deque()
    admitted = []
    for t in times:
        while log and log[0] <= t - WINDOW:
            log.popleft()
        if len(log) < LIMIT:
            log.append(t)
            admitted.append(True)
        else:
            admitted.append(False)
    return admitted

# The attack: 10 requests at :59.95 and 10 more at :00.05 - two windows, 0.1s apart.
attack = [0.995 + i * 1e-4 for i in range(10)] + [1.005 + i * 1e-4 for i in range(10)]
fixed = sum(fixed_window_admitted(attack))
exact = sum(sliding_log_admitted(attack))
print(f"20 requests within 0.1s: fixed-window admitted {fixed}, sliding log admitted {exact}")
assert fixed == 20, "fixed windows allow 2x the limit at every boundary"
assert exact == 10, "the sliding log enforces the actual contract"

The sliding log is exact but stores a timestamp per request per client — heavy at
millions of clients. Production compromise: **sliding window counter** (weight the
previous window's count by its remaining overlap) — one counter per window per client,
within a few percent of exact. Distributed deployment adds nb 05's problem: per-node
buckets over-admit N× (each node has its own budget) until you centralize the counters
(Redis) — paying a network hop per check — or accept bounded over-admission. There is no
free consistency; there never is.

### 3.4 Why the limiter exists: overload, quantified

In [ ]:
CAPACITY = 1_000                     # backend can process 1,000 req/s

def spike_day(protect: bool):
    """60s at 800 rps, a 30s spike at 3,000 rps, then calm. Queue depth over time."""
    depth, series = 0, []
    limiter = TokenBucket(capacity=1_500, refill_rate=CAPACITY * 0.95)
    for s in range(120):
        arriving = 3_000 if 60 <= s < 90 else 800
        if protect:
            arriving = sum(limiter.allow(s + i / arriving) for i in range(arriving))
        depth = max(0, depth + arriving - CAPACITY)
        series.append(depth)
    return np.array(series)

unprotected = spike_day(protect=False)
protected = spike_day(protect=True)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(unprotected, color="#c62828", label="no limiter: queue (and latency) explode")
ax.plot(protected, color="#2e7d32", label="limiter: shed to 429s, backend healthy")
ax.axvspan(60, 90, color="#ffe0b2", alpha=0.5)
ax.set_xlabel("seconds")
ax.set_ylabel("backend queue depth")
ax.set_title("A 30s spike at 3x capacity (nb 01's queueing, defended)")
ax.legend()
plt.show()

assert unprotected.max() > 50_000, "unprotected: a minute of spike = a minute of backlog"
assert protected.max() < 3_000, "protected: bounded queue, bounded latency"
assert protected[-1] == 0 and unprotected[-1] > 0, \
    "the unprotected system is STILL digesting the spike long after it ended"

That's nb 01's Little's-law lesson with a defense attached: past saturation, every
un-shed request becomes queue, and queue becomes latency for *everyone*. A 429 to one
client is a p99 kept for the rest. Load shedding is not rudeness; it's triage.

## Step 4 — Failure analysis (every arrow in the diagram)

- **Cache node dies** → miss storm hits the DB, and for a *hot* key, thousands of
  concurrent misses fetch the same code (**dogpile**). Defenses: request coalescing
  (one flight per key, others wait), soft-TTL refresh, cache replicas (nb 05).
- **DB partition unavailable** → those codes 404. Quorum-replicated partitions
  (nb 05/06) make this a blip; the cache keeps serving the hot set meanwhile.
- **ID generator clock skew** → our Snowflake refuses to mint (loud, safe — nb 08).
  Mitigation: slew clocks, per-worker sequence reserve, alert on refusals.
- **Two creates race the same custom alias** → put-if-absent decides one winner
  (nb 07's atomicity); the loser gets a clean error, never a silent overwrite (nb 05's
  LWW warning).
- **Rate-limiter store down** → fail *open* (serve without limits, alert loudly) or
  fail *closed* (429 everything)? For redirects: open — availability of other people's
  links outweighs abuse risk for minutes. For creates: closed. Per-endpoint policy,
  decided in daylight, not during the incident.

## Step 5 — Evolution

Click analytics: don't put counters in the write path — emit click *events* to a log
(nb 11) and derive counts/dashboards downstream (nb 12); the redirect path stays
read-only and fast. Expiry: tombstones + background compaction (nb 03), never
synchronous deletes. Custom domains and per-tenant limits: the rate-limiter key becomes
`(tenant, endpoint)` — and the whale-tenant problem of nb 06 ex. 1 walks right back in.

## Exercises

1. **10× the estimate.** At 1B new links/month and 300:1 reads, redo Step 2. Which
   component's design actually changes, and which just gets more replicas?
2. **301 vs 302.** A 301 (permanent) redirect lets browsers/CDNs cache the mapping —
   huge load relief. What two product capabilities does that caching quietly destroy,
   and what's the compromise?
3. **Vanity URLs.** `short.ly/nike` — custom aliases cluster alphabetically and by
   tenant. Which partitioning scheme from nb 06 handles them, and what new hot-spot
   risk do launch campaigns create?
4. **The limiter's key.** Compare rate limiting by IP, by user account, and by API key:
   give one attack or failure each misses (think NAT, account farms, key leakage).
5. **Leaky vs token.** A leaky bucket drains at a constant rate (smoothing output);
   a token bucket allows bursts. For (a) a public redirect API and (b) calls *into* a
   fragile downstream payment processor, which fits and why?

### Solutions

**1.** Writes ~390/s, reads ~117k/s avg (peak ~350k/s), storage ~180 TB. The read path
just scales out (more cache + more replicas — it was designed for this). What actually
*changes*: storage now justifies real partition/rebalancing ops (nb 06), analytics
volume forces the log-based pipeline immediately, and the cache tier becomes multi-level
(edge/CDN + regional). The single-writer ID scheme was never the bottleneck — 390/s is
still nothing.

**2.** Caching a 301 means the browser never returns: (a) click analytics vanish (the
event never reaches you), (b) link *editing/expiry* stops working (the cached mapping
outlives your change — the derived-data staleness of nb 12 in the client's pocket).
Compromise: 302 (or short-TTL 301) for links needing analytics/mutability; long 301
only for immutable enterprise links.

**3.** Hash partitioning by alias handles clustering (nb 06 — hashes don't care about
the alphabet). The new risk is a *single* alias going supernova at a product launch —
a hot key no placement fixes: cache it (local + distributed), and pre-warm before the
Super Bowl ad airs.

**4.** IP: NAT/CGNAT puts thousands of innocent users behind one IP (false limiting),
and botnets have millions of IPs (no limiting). Account: attacker farms free accounts;
also unauthenticated endpoints have no account. API key: keys leak into client-side
code and get shared; also fails the unauthenticated case. Production: layered — IP for
anonymous, key/user for authenticated, plus global circuit breakers.

**5.** (a) Public API: token bucket — human traffic is bursty; punishing a burst of 5
clicks feels broken; cap the *average*. (b) Fragile downstream: leaky bucket (constant
drain) — the processor cares about instantaneous rate; you want smoothing, not burst
tolerance. Same math family, opposite goals: protect *your* budget vs protect *their*
ingest.

## Takeaways

- The estimate wrote the architecture: tiny writes, Zipf-hot reads, small data — so
  ID scheme + cache + partitioning are the whole game, and effort goes where the
  numbers point.
- ID generation is a three-way trade: sequential (compact, enumerable, hot-tailed) vs
  snowflake (coordination-free, time-sorted, clock-dependent) vs random (unguessable,
  uniform, needs put-if-absent). All three met Part 1 on the way through.
- Zipf skew makes small caches mighty — and neutralizes the un-partitionable hot key.
- Rate limiting is load-bearing architecture: token buckets for bursty averages,
  sliding windows for strict contracts, fixed windows for a 2× hole; shedding early is
  how the p99 survives spikes.
- Every failure answer came from a Part 1 notebook. That was the point.

**Next:** [23 · News feed](23-capstone-news-feed.ipynb) — the read path fights back:
fan-out, celebrities, and the write-time/read-time bargain at social scale.